# LENS / Pythia-31M: Joint LearnedNorm + Direct PureMLP Attention on The Pile

This notebook is a **minimal language-model adaptation** of the current LENS recovery workflow.

The experiment deliberately keeps the procedure simple:

1. download/load the final pretrained `EleutherAI/pythia-31m` checkpoint (`step143000`, identical to `main`);
2. build a fixed local subset from the original **Pile training split** and a fixed Pile validation subset;
3. evaluate the untouched pretrained Pythia-31M on the held-out Pile validation subset;
4. replace **all LayerNorm modules with LearnedNorm**;
5. replace **all causal self-attention Softmax mappings with Direct PureMLP attention**;
6. install both operator families **at the same time**;
7. unfreeze the whole model immediately;
8. retrain end-to-end on the Pile causal-language-modeling objective for **50 epochs**;
9. compare validation LM loss, perplexity, and next-token accuracy before replacement, immediately after replacement, and after recovery.

There is no QNLI/downstream fine-tuning in this notebook.

## Fixed sequence length

For a manageable pilot, sequences are packed to:

\[
N=512.
\]

Pythia supports longer contexts, but the current PureMLP operator has a row MLP whose input dimension equals the sequence length, so this experiment intentionally uses the same compact fixed-length setup as the preceding tests.

## Causal valid-count scaling

For every causal query row \(q\), only its valid keys are counted:

\[
L_q=\sum_j M_{qj}.
\]

The gain divisor is independently computed for every row:

\[
C_q=2^{\lfloor\log_2 L_q\rfloor}.
\]

Thus:

\[
A_{qj}=
\frac{\sigma(u_{qj})}{C_q}
\]

for valid causal positions, while future/masked positions are forced to zero.

No attention Softmax, `exp`, or `exp2` is used.

## Local storage

- model: `model/pythia31m`
- packed Pile data: `data/Pile`

The full Pile is extremely large, so this pilot streams only enough **real Pile training/validation text** to create a fixed local packed subset. Once the packed tensors exist locally, later runs do not redownload or retokenize them.


In [1]:
# The current environment used for the BERT experiment is already suitable:
#   torch == 2.1.0
#   transformers == 4.36.2
#   datasets == 3.6.0
#   huggingface_hub == 0.36.2
#
# If these packages are missing in another environment, install compatible
# versions before running the rest of the notebook.


In [2]:
# ============================================================
# Imports and configuration
# ============================================================

import json
import math
import os
import random
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.gpt_neox.modeling_gpt_neox import GPTNeoXAttention


# ------------------------------------------------------------
# Reproducibility / runtime
# ------------------------------------------------------------
SEED = 42
NUM_WORKERS = 8
BATCH_SIZE = 16
PIN_MEMORY = True

torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

USE_AMP = True
AMP_DTYPE = torch.bfloat16

# None -> current/default visible CUDA device.
# 0/1/... -> explicit logical CUDA index.
CUDA_DEVICE_INDEX = 1


# ------------------------------------------------------------
# Pythia-31M
# ------------------------------------------------------------
MODEL_REPO = "EleutherAI/pythia-31m"

# Official final checkpoint. The Pythia model card states that
# step143000 is identical to the main branch.
MODEL_REVISION = "step143000"

MODEL_DIR = Path("model/pythia31m")


# ------------------------------------------------------------
# Pile
# ------------------------------------------------------------
PILE_REPO = "EleutherAI/pile-duped-pythia-random-sampled"

DATA_DIR = Path("data/Pile")
HF_DATA_CACHE = DATA_DIR / "hf_cache"

SEQ_LENGTH = 512

# Pilot subset sizes.
#
# These are packed fixed-length causal-LM sequences sampled from the real
# Pile train/validation streams. Increase them later if desired.

TRAIN_SEQUENCES = 80_000
VAL_SEQUENCES = 5_000

TRAIN_TENSOR_PATH = (
    DATA_DIR
    / f"pile_train_packed_{SEQ_LENGTH}_{TRAIN_SEQUENCES}.pt"
)

VAL_TENSOR_PATH = (
    DATA_DIR
    / f"pile_validation_packed_{SEQ_LENGTH}_{VAL_SEQUENCES}.pt"
)

DATA_META_PATH = DATA_DIR / (
    f"pile_packed_{SEQ_LENGTH}_"
    f"{TRAIN_SEQUENCES}_{VAL_SEQUENCES}_meta.json"
)


# ------------------------------------------------------------
# LENS modules
# ------------------------------------------------------------
LEARNED_NORM_HIDDEN = 32

ATTENTION_ROW_HIDDEN = 32
ATTENTION_ELEMENT_HIDDEN = 16

LEARNED_NORM_A_MIN = 0.05
LEARNED_NORM_A_MAX = 12.0


# ------------------------------------------------------------
# Full-LENS recovery
# ------------------------------------------------------------
RECOVERY_EPOCHS = 50
RECOVERY_LR = 2.5e-4

WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0

RUN_VARIANT = "pythia31m_pile_full_lens"

OUT_DIR = Path(
    "./runs/lens_pythia31m_pile_full_lens"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = OUT_DIR / "best.pt"
LAST_CKPT = OUT_DIR / "last.pt"
LOG_FILE = OUT_DIR / "training_log.txt"
HISTORY_FILE = OUT_DIR / "history.json"

AUTO_RESUME = True


/home/yuhao/anaconda3/envs/brevitas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# Reproducibility and selected device
# ============================================================

def seed_cpu_side(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_cpu_side(SEED)

if torch.cuda.is_available():
    if CUDA_DEVICE_INDEX is None:
        selected_cuda_index = torch.cuda.current_device()
    else:
        selected_cuda_index = int(CUDA_DEVICE_INDEX)

        if (
            selected_cuda_index < 0
            or selected_cuda_index >= torch.cuda.device_count()
        ):
            raise ValueError(
                f"CUDA_DEVICE_INDEX={selected_cuda_index} is invalid; "
                f"{torch.cuda.device_count()} CUDA device(s) are visible."
            )

    torch.cuda.set_device(selected_cuda_index)

    device = torch.device(
        f"cuda:{selected_cuda_index}"
    )

    torch.cuda.manual_seed_all(SEED)

    print(
        "Selected CUDA device:",
        device,
        "|",
        torch.cuda.get_device_name(
            selected_cuda_index
        ),
    )
else:
    device = torch.device("cpu")
    print("CUDA is unavailable; using CPU.")


print("PyTorch:", torch.__version__)
print("Device:", device)


Selected CUDA device: cuda:1 | NVIDIA A40
PyTorch: 2.1.0
Device: cuda:1


## 1. Download / load the final pretrained Pythia-31M checkpoint

The model is stored in:

`model/pythia31m`

The notebook first checks whether the local model files already exist.

- If the local snapshot is complete, download is skipped.
- Otherwise, `EleutherAI/pythia-31m` revision `step143000` is downloaded once.
- The tokenizer and model are then loaded with `local_files_only=True`.
- KV cache is disabled during recovery training.


In [4]:
# ============================================================
# Download / load pretrained Pythia-31M
# ============================================================

MODEL_DIR.mkdir(parents=True, exist_ok=True)


def local_pythia_is_complete(model_dir):
    model_dir = Path(model_dir)

    has_config = (
        model_dir / "config.json"
    ).exists()

    has_tokenizer = any(
        (model_dir / name).exists()
        for name in [
            "tokenizer.json",
            "tokenizer_config.json",
            "vocab.json",
        ]
    )

    has_weights = any(
        (model_dir / name).exists()
        for name in [
            "model.safetensors",
            "pytorch_model.bin",
        ]
    )

    # Also support sharded checkpoints.
    has_weight_index = any(
        (model_dir / name).exists()
        for name in [
            "model.safetensors.index.json",
            "pytorch_model.bin.index.json",
        ]
    )

    return (
        has_config
        and has_tokenizer
        and (has_weights or has_weight_index)
    )


if local_pythia_is_complete(MODEL_DIR):
    print(
        "Local Pythia-31M found. Skip download:",
        MODEL_DIR.resolve(),
    )
else:
    print(
        "Local Pythia-31M is missing/incomplete. "
        "Downloading revision",
        MODEL_REVISION,
        "to:",
        MODEL_DIR.resolve(),
    )

    snapshot_download(
        repo_id=MODEL_REPO,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    )

    if not local_pythia_is_complete(MODEL_DIR):
        raise FileNotFoundError(
            "Pythia-31M download finished, but the expected local files "
            f"were not found under {MODEL_DIR.resolve()}."
        )


tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)

if tokenizer.eos_token_id is None:
    raise ValueError(
        "Pythia tokenizer has no EOS token id; "
        "the packing code expects one document separator token."
    )


def load_local_pythia31m():
    common_kwargs = dict(
        pretrained_model_name_or_path=str(MODEL_DIR),
        local_files_only=True,
    )

    try:
        model = AutoModelForCausalLM.from_pretrained(
            **common_kwargs,
            attn_implementation="eager",
        )
    except TypeError:
        # Compatibility with Transformers 4.36.2.
        model = AutoModelForCausalLM.from_pretrained(
            **common_kwargs,
        )

    model.config.use_cache = False
    return model


model = load_local_pythia31m()

print("\nModel summary:")
print("  repository:", MODEL_REPO)
print("  revision:", MODEL_REVISION)
print("  local path:", MODEL_DIR.resolve())
print(
    "  blocks:",
    model.config.num_hidden_layers,
)
print(
    "  hidden size:",
    model.config.hidden_size,
)
print(
    "  heads:",
    model.config.num_attention_heads,
)
print(
    "  max positions:",
    model.config.max_position_embeddings,
)
print(
    "  experiment sequence length:",
    SEQ_LENGTH,
)
print(
    "  parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)

assert hasattr(model, "gpt_neox")
assert hasattr(model.gpt_neox, "layers")

assert (
    len(model.gpt_neox.layers)
    == model.config.num_hidden_layers
)

assert (
    SEQ_LENGTH
    <= model.config.max_position_embeddings
)

for i, layer in enumerate(
    model.gpt_neox.layers
):
    assert hasattr(
        layer,
        "attention",
    ), f"layer {i} has no attention"

    assert isinstance(
        layer.attention,
        GPTNeoXAttention,
    ), (
        f"layer {i}.attention is "
        f"{type(layer.attention).__name__}, "
        "but this notebook expects eager GPTNeoXAttention."
    )


Local Pythia-31M is missing/incomplete. Downloading revision step143000 to: /home/yuhao/project/LENS/Pythia/seed42/model/pythia31m


Fetching 9 files: 100%|███████████████████████████| 9/9 [00:06<00:00,  1.38it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



Model summary:
  repository: EleutherAI/pythia-31m
  revision: step143000
  local path: /home/yuhao/project/LENS/Pythia/seed42/model/pythia31m
  blocks: 6
  hidden size: 256
  heads: 8
  max positions: 2048
  experiment sequence length: 512
  parameters: 30,494,720


## 2. Build a fixed local Pythia-training-data subset

The standard `EleutherAI/pile` loading script currently points to the old
`the-eye.eu` host.  Instead, this notebook uses EleutherAI's HF-hosted
**pre-tokenized random sample from the standard ("duped") Pythia training
data**:

`EleutherAI/pile-duped-pythia-random-sampled`

This avoids the external `the-eye.eu` server entirely.

The sampled dataset already contains Pythia token IDs in its `Tokens`
column, so there is no second tokenizer pass.

The notebook streams only enough rows to build the local pilot tensors,
then stores them under:

`data/Pile`

Default local pilot:

- train: `64,000 × 512` tokens
- validation: `5,000 × 512` tokens

To make fixed 512-token sequences robustly even if a sampled row is shorter
than 512 tokens, sampled token lists are concatenated with one EOS token
between rows and then packed into contiguous 512-token blocks.

Once the packed tensors exist, later runs load them locally and skip the
Hugging Face stream.


In [5]:
# ============================================================
# HF-hosted Pythia training-data sample -> fixed packed tensors
# ============================================================

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HF_DATA_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)


def open_pythia_sample_stream():
    """
    EleutherAI-hosted random sample of the standard ("duped") Pythia
    training data.

    It is already pre-tokenized and exposes token IDs in the `Tokens`
    column.  Streaming reads the HF-hosted parquet data directly and does
    not touch the old the-eye.eu Pile URLs.
    """
    return load_dataset(
        PILE_REPO,
        split="train",
        streaming=True,
        cache_dir=str(HF_DATA_CACHE),
    )


def build_all_packed_tensors():
    """
    Build TRAIN_SEQUENCES + VAL_SEQUENCES fixed-length blocks in one pass.

    We deliberately create train and validation from disjoint portions of
    the random-sampled Pythia training-data stream:
        first TRAIN_SEQUENCES blocks -> train
        next  VAL_SEQUENCES blocks   -> validation

    The validation subset here is therefore a held-out local recovery
    subset from the same Pythia-training-data sample.  It is used only for
    comparing standard / immediate-replacement / recovered models under
    exactly the same distribution.
    """

    if TRAIN_TENSOR_PATH.exists() and VAL_TENSOR_PATH.exists():
        print(
            "Local packed Pythia/Pile tensors found. "
            "Skip download/packing."
        )

        train_tensor = torch.load(
            TRAIN_TENSOR_PATH,
            map_location="cpu",
        )

        val_tensor = torch.load(
            VAL_TENSOR_PATH,
            map_location="cpu",
        )

        expected_train = (
            TRAIN_SEQUENCES,
            SEQ_LENGTH,
        )

        expected_val = (
            VAL_SEQUENCES,
            SEQ_LENGTH,
        )

        if tuple(train_tensor.shape) != expected_train:
            raise ValueError(
                f"Cached train tensor shape {tuple(train_tensor.shape)} "
                f"does not match expected {expected_train}."
            )

        if tuple(val_tensor.shape) != expected_val:
            raise ValueError(
                f"Cached val tensor shape {tuple(val_tensor.shape)} "
                f"does not match expected {expected_val}."
            )

        return train_tensor, val_tensor

    print(
        "Streaming:",
        PILE_REPO,
    )

    stream = open_pythia_sample_stream()

    eos_id = int(
        tokenizer.eos_token_id
    )

    total_needed = (
        TRAIN_SEQUENCES
        + VAL_SEQUENCES
    )

    packed = []
    token_buffer = []
    cursor = 0
    rows_seen = 0

    for example in stream:
        tokens = example.get(
            "Tokens",
            None,
        )

        if tokens is None:
            continue

        # HF may expose the sequence as list / ndarray / Arrow-backed object.
        tokens = [
            int(x)
            for x in tokens
        ]

        if not tokens:
            continue

        # Preserve a clear boundary between independently sampled sequences.
        token_buffer.extend(
            tokens
        )
        token_buffer.append(
            eos_id
        )

        rows_seen += 1

        while (
            len(token_buffer) - cursor
            >= SEQ_LENGTH
            and len(packed) < total_needed
        ):
            packed.append(
                token_buffer[
                    cursor : cursor + SEQ_LENGTH
                ]
            )

            cursor += (
                SEQ_LENGTH
            )

        # Discard already-consumed prefix occasionally.
        if cursor >= 1_000_000:
            token_buffer = (
                token_buffer[cursor:]
            )
            cursor = 0

        if len(packed) >= total_needed:
            break

        if rows_seen % 10_000 == 0:
            print(
                f"  sampled rows={rows_seen:,} | "
                f"packed={len(packed):,}/{total_needed:,}"
            )

    if len(packed) < total_needed:
        raise RuntimeError(
            f"Stream ended after producing only "
            f"{len(packed):,}/{total_needed:,} packed sequences."
        )

    all_tensor = torch.tensor(
        packed,
        dtype=torch.int32,
    )

    train_tensor = (
        all_tensor[
            :TRAIN_SEQUENCES
        ]
        .contiguous()
    )

    val_tensor = (
        all_tensor[
            TRAIN_SEQUENCES:
            TRAIN_SEQUENCES + VAL_SEQUENCES
        ]
        .contiguous()
    )

    torch.save(
        train_tensor,
        TRAIN_TENSOR_PATH,
    )

    torch.save(
        val_tensor,
        VAL_TENSOR_PATH,
    )

    print(
        "Saved train:",
        TRAIN_TENSOR_PATH.resolve(),
        "| shape=",
        tuple(train_tensor.shape),
    )

    print(
        "Saved validation:",
        VAL_TENSOR_PATH.resolve(),
        "| shape=",
        tuple(val_tensor.shape),
    )

    print(
        "Source sampled rows consumed:",
        f"{rows_seen:,}",
    )

    return (
        train_tensor,
        val_tensor,
    )


train_tokens, val_tokens = (
    build_all_packed_tensors()
)


metadata = {
    "dataset_repo": PILE_REPO,
    "dataset_kind": "EleutherAI HF-hosted random sample from standard/duped Pythia training data",
    "model_repo": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "sequence_length": SEQ_LENGTH,
    "train_sequences": TRAIN_SEQUENCES,
    "validation_sequences": VAL_SEQUENCES,
    "eos_token_id": int(tokenizer.eos_token_id),
}

with DATA_META_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )


class PackedCausalLMDataset(Dataset):
    def __init__(self, token_tensor):
        if (
            token_tensor.ndim != 2
            or token_tensor.shape[1] != SEQ_LENGTH
        ):
            raise ValueError(
                f"Expected [N,{SEQ_LENGTH}], "
                f"got {tuple(token_tensor.shape)}."
            )

        self.tokens = token_tensor

    def __len__(self):
        return self.tokens.shape[0]

    def __getitem__(self, idx):
        return self.tokens[idx]


train_set = PackedCausalLMDataset(
    train_tokens
)

val_set = PackedCausalLMDataset(
    val_tokens
)


def causal_lm_collate(batch):
    input_ids = torch.stack(
        batch,
        dim=0,
    ).long()

    attention_mask = torch.ones_like(
        input_ids,
        dtype=torch.long,
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone(),
    }


loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=causal_lm_collate,
)

if NUM_WORKERS > 0:
    loader_kwargs[
        "persistent_workers"
    ] = True


train_loader = DataLoader(
    train_set,
    shuffle=True,
    drop_last=False,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_set,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)


print("\nDataset summary:")
print(
    "  source:",
    PILE_REPO,
)
print(
    "  train sequences:",
    len(train_set),
)
print(
    "  validation sequences:",
    len(val_set),
)
print(
    "  train batches:",
    len(train_loader),
)
print(
    "  validation batches:",
    len(val_loader),
)
print(
    "  fixed sequence length:",
    SEQ_LENGTH,
)
print(
    "  train tokens per epoch:",
    f"{len(train_set) * SEQ_LENGTH:,}",
)


Local packed Pythia/Pile tensors found. Skip download/packing.

Dataset summary:
  source: EleutherAI/pile-duped-pythia-random-sampled
  train sequences: 64000
  validation sequences: 5000
  train batches: 4000
  validation batches: 313
  fixed sequence length: 512
  train tokens per epoch: 32,768,000


## 3. Learned replacements

### LearnedNorm

The same dynamic affine replacement is used:

\[
[a_t,b_t]=f_\theta(x_t),
\qquad
y_t=a_t x_t+b_t.
\]

with:

\[
D\rightarrow32\rightarrow2.
\]

`a_t` is clamped to `[0.05, 12]`; `b_t` is not clamped.

### Direct PureMLP causal attention

For each complete fixed-length score row:

\[
Z_q\in\mathbb{R}^{512}
\rightarrow
512\rightarrow32\rightarrow1
\rightarrow s_q.
\]

Then for every score:

\[
[z_{qj},s_q]
\rightarrow
2\rightarrow16\rightarrow1
\rightarrow u_{qj}.
\]

For every causal query row independently:

\[
L_q = \#\{\text{valid keys for query }q\},
\]

\[
C_q=2^{\lfloor\log_2L_q\rfloor},
\]

\[
A_{qj}
=
\frac{\sigma(u_{qj})}{C_q}.
\]

Future/masked keys contribute exactly zero.


In [6]:
# ============================================================
# Learned LayerNorm replacement
# ============================================================

class LearnedLayerNormReplacement(nn.Module):
    """
    Dynamic affine LayerNorm replacement:
        [a_t, b_t] = MLP(x_t)
        y_t = a_t * x_t + b_t
    """

    def __init__(
        self,
        dim,
        hidden_dim=32,
        name="ln",
    ):
        super().__init__()

        self.dim = int(dim)
        self.name = name

        self.predictor_0 = nn.Linear(
            self.dim,
            hidden_dim,
            bias=True,
        )

        self.predictor_relu = nn.ReLU()

        self.predictor_1 = nn.Linear(
            hidden_dim,
            2,
            bias=True,
        )

        nn.init.kaiming_uniform_(
            self.predictor_0.weight,
            nonlinearity="relu",
        )

        nn.init.zeros_(
            self.predictor_0.bias
        )

        nn.init.normal_(
            self.predictor_1.weight,
            mean=0.0,
            std=1e-3,
        )

        with torch.no_grad():
            self.predictor_1.bias[0] = 0.5
            self.predictor_1.bias[1] = 0.0

    def forward(
        self,
        x,
    ):
        state = self.predictor_0(x)
        state = self.predictor_relu(
            state
        )
        state = self.predictor_1(
            state
        )

        raw_a = state[..., 0:1]
        b = state[..., 1:2]

        a = torch.clamp(
            raw_a,
            min=LEARNED_NORM_A_MIN,
            max=LEARNED_NORM_A_MAX,
        )

        return a * x + b


# ============================================================
# Direct pure-MLP attention replacement
# ============================================================

class DirectMLPAttentionReplacement(nn.Module):
    """
    scores: [B, H, N_query, N_key]

    N_key is fixed at SEQ_LENGTH.

    Each query row receives its own valid-key count and therefore its own
    power-of-two gain divisor:
        C_q = 2 ** floor(log2(valid_count_q))
    """

    def __init__(
        self,
        num_tokens,
        row_hidden_dim=32,
        element_hidden_dim=16,
    ):
        super().__init__()

        self.num_tokens = int(
            num_tokens
        )

        self.row_state_mlp = nn.Sequential(
            nn.Linear(
                self.num_tokens,
                row_hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                row_hidden_dim,
                1,
            ),
        )

        self.element_mlp = nn.Sequential(
            nn.Linear(
                2,
                element_hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                element_hidden_dim,
                1,
            ),
        )

        for module in self.modules():
            if isinstance(
                module,
                nn.Linear,
            ):
                nn.init.trunc_normal_(
                    module.weight,
                    std=0.02,
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

    def forward(
        self,
        scores,
        valid_key_mask,
    ):
        if scores.ndim != 4:
            raise ValueError(
                f"Expected scores [B,H,Nq,Nk], "
                f"got {tuple(scores.shape)}."
            )

        B, H, Nq, Nk = (
            scores.shape
        )

        if Nk != self.num_tokens:
            raise ValueError(
                f"PureMLP expects N_key={self.num_tokens}, "
                f"but received {Nk}."
            )

        valid_key_mask = (
            valid_key_mask
            .to(
                device=scores.device,
                dtype=torch.bool,
            )
        )

        # Invalid/future positions are exactly zero before both MLPs.
        masked_scores = (
            scores.masked_fill(
                ~valid_key_mask,
                0.0,
            )
        )

        # ----------------------------------------------------
        # 1) One learned scalar state per complete query row.
        # ----------------------------------------------------
        rows = masked_scores.reshape(
            B * H * Nq,
            Nk,
        )

        row_state = (
            self.row_state_mlp(
                rows
            )
            .reshape(
                B,
                H,
                Nq,
                1,
            )
        )

        # ----------------------------------------------------
        # 2) Shared element mapping.
        # ----------------------------------------------------
        expanded_state = (
            row_state.expand(
                -1,
                -1,
                -1,
                Nk,
            )
        )

        element_input = torch.stack(
            [
                masked_scores,
                expanded_state,
            ],
            dim=-1,
        )

        logits = (
            self.element_mlp(
                element_input
            )
            .squeeze(-1)
        )

        # ----------------------------------------------------
        # 3) Per-row power-of-two valid-count divisor.
        #
        # valid_key_mask can have broadcast dimensions, e.g.
        # [1,1,Nq,Nk] for a pure causal mask or [B,1,Nq,Nk]
        # if padding is also present.
        # ----------------------------------------------------
        valid_count = (
            valid_key_mask
            .sum(
                dim=-1,
                keepdim=True,
            )
            .to(
                dtype=scores.dtype
            )
        )

        valid_count = torch.clamp(
            valid_count,
            min=1.0,
        )

        gain_divisor = torch.pow(
            2.0,
            torch.floor(
                torch.log2(
                    valid_count
                )
            ),
        )

        attn = (
            torch.sigmoid(logits)
            / gain_divisor
        )

        attn = (
            attn
            * valid_key_mask.to(
                attn.dtype
            )
        )

        return attn


## 4. GPT-NeoX attention replacement

Pythia-31M uses Hugging Face `GPTNeoXAttention`.

To keep the modification minimal and preserve the pretrained Q/K/V projections, RoPE, output projection, cache interface, and block structure, the replacement class **inherits the original Transformers 4.36.2 `GPTNeoXAttention` implementation and overrides only `_attn()`**.

The original `_attn()` computes:

\[
QK^T/\sqrt{d}
\rightarrow
\text{causal mask}
\rightarrow
\text{Softmax}.
\]

The new `_attn()` keeps the same scaled \(QK^T\) scores and causal mask, but sends the masked score rows into Direct PureMLP instead of Softmax.


In [7]:
# ============================================================
# GPT-NeoX attention with Direct PureMLP
# Compatible with transformers == 4.36.2
# ============================================================

class GPTNeoXAttentionWithDirectMLP(
    GPTNeoXAttention
):
    """
    Preserve the stock GPTNeoXAttention forward path and all pretrained
    projections, but replace only its _attn() score-to-weight mapping.
    """

    def __init__(
        self,
        original_attn,
        num_tokens,
        row_hidden_dim=32,
        element_hidden_dim=16,
    ):
        # Build the same stock attention module.
        super().__init__(
            original_attn.config
        )

        # Copy all pretrained attention parameters before adding the new
        # PureMLP parameters.
        self.load_state_dict(
            original_attn.state_dict(),
            strict=True,
        )

        self.learned_attention = (
            DirectMLPAttentionReplacement(
                num_tokens=num_tokens,
                row_hidden_dim=row_hidden_dim,
                element_hidden_dim=element_hidden_dim,
            )
        )

        # Preserve train/eval state.
        self.train(
            original_attn.training
        )

    def _attn(
        self,
        query,
        key,
        value,
        attention_mask=None,
        head_mask=None,
    ):
        # q/k/v:
        # [B, H, N, head_dim]

        (
            batch_size,
            num_attention_heads,
            query_length,
            attn_head_size,
        ) = query.size()

        key_length = key.size(-2)

        if key_length != SEQ_LENGTH:
            raise ValueError(
                f"This fixed-length PureMLP experiment expects "
                f"key_length={SEQ_LENGTH}, got {key_length}. "
                "Disable generation/KV-cache paths during recovery."
            )

        if query_length != SEQ_LENGTH:
            raise ValueError(
                f"This fixed-length PureMLP experiment expects "
                f"query_length={SEQ_LENGTH}, got {query_length}."
            )

        # Dynamically grow causal mask buffer if needed.
        if key_length > self.bias.shape[-1]:
            self._init_bias(
                key_length,
                device=key.device,
            )

        causal_mask = self.bias[
            :,
            :,
            key_length - query_length : key_length,
            :key_length,
        ]

        # Same scaled QK^T computation as stock GPT-NeoX.
        query_2d = query.view(
            batch_size * num_attention_heads,
            query_length,
            attn_head_size,
        )

        key_2d = key.view(
            batch_size * num_attention_heads,
            key_length,
            attn_head_size,
        )

        attn_scores = torch.zeros(
            batch_size * num_attention_heads,
            query_length,
            key_length,
            dtype=query.dtype,
            device=key.device,
        )

        attn_scores = torch.baddbmm(
            attn_scores,
            query_2d,
            key_2d.transpose(1, 2),
            beta=1.0,
            alpha=self.norm_factor,
        )

        attn_scores = attn_scores.view(
            batch_size,
            num_attention_heads,
            query_length,
            key_length,
        )

        # ----------------------------------------------------
        # Build exact valid-key mask.
        #
        # causal_mask:
        #   [1,1,Nq,Nk]
        #
        # attention_mask from GPTNeoXModel, when present:
        #   additive mask where 0 is valid and a large negative value
        #   is masked/padding.
        # ----------------------------------------------------
        valid_key_mask = causal_mask

        if attention_mask is not None:
            additive_valid = (
                attention_mask
                >= -1e-6
            )

            valid_key_mask = (
                valid_key_mask
                & additive_valid
            )

        attn_weights = (
            self.learned_attention(
                attn_scores,
                valid_key_mask=valid_key_mask,
            )
        )

        if head_mask is not None:
            attn_weights = (
                attn_weights
                * head_mask
            )

        attn_weights = (
            self.attention_dropout(
                attn_weights
            )
        )

        attn_output = torch.matmul(
            attn_weights,
            value,
        )

        return (
            attn_output,
            attn_weights,
        )


## 5. Automatic replacement helpers

LayerNorm replacement recursively walks the entire model, so this replaces:

- every block input LayerNorm;
- every block post-attention LayerNorm;
- the final GPT-NeoX LayerNorm.

Attention replacement changes:

`gpt_neox.layers[*].attention`

for all Pythia blocks.

Both operator families are installed in the same model before recovery training.


In [8]:
# ============================================================
# Automatic replacement helpers
# ============================================================

def _is_layernorm(
    module,
):
    return isinstance(
        module,
        nn.LayerNorm,
    )


def replace_all_layernorms(
    module,
    hidden_dim=32,
    prefix="",
    replaced=None,
):
    if replaced is None:
        replaced = []

    for child_name, child in list(
        module.named_children()
    ):
        full_name = (
            f"{prefix}.{child_name}"
            if prefix
            else child_name
        )

        if isinstance(
            child,
            LearnedLayerNormReplacement,
        ):
            continue

        if _is_layernorm(
            child
        ):
            normalized_shape = (
                child.normalized_shape
            )

            if isinstance(
                normalized_shape,
                int,
            ):
                dim = normalized_shape
            else:
                if (
                    len(normalized_shape)
                    != 1
                ):
                    raise ValueError(
                        f"Only 1-D LayerNorm is supported; "
                        f"got {normalized_shape} at {full_name}."
                    )

                dim = int(
                    normalized_shape[0]
                )

            setattr(
                module,
                child_name,
                LearnedLayerNormReplacement(
                    dim=dim,
                    hidden_dim=hidden_dim,
                    name=full_name,
                ),
            )

            replaced.append(
                full_name
            )

        else:
            replace_all_layernorms(
                child,
                hidden_dim=hidden_dim,
                prefix=full_name,
                replaced=replaced,
            )

    return replaced


def replace_all_attention_mapping(
    model,
    row_hidden_dim=32,
    element_hidden_dim=16,
):
    replaced = []

    for layer_id, layer in enumerate(
        model.gpt_neox.layers
    ):
        if isinstance(
            layer.attention,
            GPTNeoXAttentionWithDirectMLP,
        ):
            continue

        if not isinstance(
            layer.attention,
            GPTNeoXAttention,
        ):
            raise TypeError(
                f"Layer {layer_id} attention is "
                f"{type(layer.attention).__name__}; "
                "expected eager GPTNeoXAttention."
            )

        layer.attention = (
            GPTNeoXAttentionWithDirectMLP(
                original_attn=layer.attention,
                num_tokens=SEQ_LENGTH,
                row_hidden_dim=row_hidden_dim,
                element_hidden_dim=element_hidden_dim,
            )
        )

        replaced.append(
            f"gpt_neox.layers.{layer_id}.attention"
        )

    return replaced


def unfreeze_all(
    model,
):
    for p in model.parameters():
        p.requires_grad = True


def parameter_counts(
    model,
):
    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return (
        trainable,
        total,
    )


def print_trainable_parameters(
    model,
    title="",
):
    trainable, total = (
        parameter_counts(
            model
        )
    )

    if title:
        print(title)

    print(
        f"Trainable: {trainable:,} / {total:,} "
        f"({100.0 * trainable / total:.4f}%)"
    )


## 6. Causal-LM evaluation / training / logging / resume

Metrics:

- **validation LM loss** — primary recovery metric;
- **perplexity** \(=\exp(\text{loss})\);
- **next-token accuracy** — direct token prediction accuracy.

The best checkpoint is selected by **lowest validation LM loss**.

Training keeps the same simple recovery style:

- AdamW;
- cosine learning-rate decay;
- full-model joint training;
- gradient clipping;
- `best.pt`;
- `last.pt`;
- automatic resume;
- text log and JSON history.


In [9]:
# ============================================================
# Causal-LM evaluation / training / checkpoint helpers
# ============================================================

def autocast_context(
    device,
):
    if (
        USE_AMP
        and device.type == "cuda"
    ):
        return torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        )

    return nullcontext()


def sync_device(
    device,
):
    if device.type == "cuda":
        torch.cuda.synchronize(
            device
        )


def format_seconds(
    seconds,
):
    seconds = float(
        seconds
    )

    if seconds < 60:
        return (
            f"{seconds:.1f}s"
        )

    minutes, sec = divmod(
        seconds,
        60,
    )

    if minutes < 60:
        return (
            f"{int(minutes)}m "
            f"{sec:.1f}s"
        )

    hours, minutes = divmod(
        minutes,
        60,
    )

    return (
        f"{int(hours)}h "
        f"{int(minutes)}m "
        f"{sec:.1f}s"
    )


def append_log(
    path,
    text,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            text.rstrip()
            + "\n"
        )


def move_batch_to_device(
    batch,
    device,
):
    return {
        key: value.to(
            device,
            non_blocking=True,
        )
        for key, value
        in batch.items()
    }


def safe_perplexity(
    loss,
):
    # Avoid inf in diagnostic printing.
    return math.exp(
        min(
            float(loss),
            20.0,
        )
    )


@torch.no_grad()
def evaluate_lm(
    model,
    loader,
    device,
):
    model.eval()

    total_target_tokens = 0
    total_correct = 0
    loss_sum = 0.0

    sync_device(
        device
    )

    start = time.perf_counter()

    for batch in loader:
        batch = move_batch_to_device(
            batch,
            device,
        )

        with autocast_context(
            device
        ):
            outputs = model(
                **batch,
                use_cache=False,
            )

            loss = outputs.loss
            logits = outputs.logits

        # Causal LM predicts token t+1 from positions <= t.
        shifted_logits = logits[
            :,
            :-1,
            :,
        ]

        shifted_labels = batch[
            "labels"
        ][
            :,
            1:,
        ]

        predictions = (
            shifted_logits.argmax(
                dim=-1
            )
        )

        target_tokens = (
            shifted_labels.numel()
        )

        total_target_tokens += (
            target_tokens
        )

        total_correct += (
            predictions
            == shifted_labels
        ).sum().item()

        loss_sum += (
            float(loss.item())
            * target_tokens
        )

    sync_device(
        device
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    mean_loss = (
        loss_sum
        / total_target_tokens
    )

    token_accuracy = (
        total_correct
        / total_target_tokens
    )

    return {
        "loss": mean_loss,
        "perplexity": safe_perplexity(
            mean_loss
        ),
        "token_accuracy": token_accuracy,
        "target_tokens": total_target_tokens,
        "time_sec": elapsed,
    }


def print_eval(
    model,
    loader,
    device,
    title="",
    split_name="Validation",
):
    if title:
        print(
            f"\n[{title}]"
        )

    metrics = evaluate_lm(
        model,
        loader,
        device,
    )

    print(
        f"{split_name}: "
        f"loss={metrics['loss']:.4f}, "
        f"ppl={metrics['perplexity']:.3f}, "
        f"next-token acc={metrics['token_accuracy']:.4f}, "
        f"time={format_seconds(metrics['time_sec'])}"
    )

    return metrics


def portable_torch_load(
    path,
):
    return torch.load(
        path,
        map_location="cpu",
    )


def atomic_torch_save(
    obj,
    path,
):
    path = Path(path)

    temp_path = (
        path.parent
        / (
            path.name
            + ".tmp"
        )
    )

    torch.save(
        obj,
        temp_path,
    )

    os.replace(
        temp_path,
        path,
    )


def checkpoint_config():
    return {
        "run_variant": RUN_VARIANT,
        "model_repo": MODEL_REPO,
        "model_revision": MODEL_REVISION,
        "sequence_length": SEQ_LENGTH,
        "train_sequences": TRAIN_SEQUENCES,
        "validation_sequences": VAL_SEQUENCES,
        "learned_norm_hidden": LEARNED_NORM_HIDDEN,
        "attention_row_hidden": ATTENTION_ROW_HIDDEN,
        "attention_element_hidden": ATTENTION_ELEMENT_HIDDEN,
        "recovery_lr": RECOVERY_LR,
        "weight_decay": WEIGHT_DECAY,
    }


def load_model_checkpoint(
    model,
    path,
    device,
    strict=True,
):
    ckpt = portable_torch_load(
        path
    )

    saved_config = ckpt.get(
        "config",
        {},
    )

    expected = checkpoint_config()

    for key, value in expected.items():
        if (
            key in saved_config
            and saved_config[key] != value
        ):
            raise ValueError(
                f"Checkpoint config mismatch for '{key}': "
                f"saved={saved_config[key]!r}, current={value!r}"
            )

    model.load_state_dict(
        ckpt["model"],
        strict=strict,
    )

    model.to(
        device
    )

    print(
        f"Loaded checkpoint: {path} | "
        f"epoch={ckpt.get('epoch')} | "
        f"val_loss={ckpt.get('val', {}).get('loss')} | "
        f"current device={device}"
    )

    return ckpt


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs,
    lr,
    weight_decay,
    grad_clip_norm=1.0,
    auto_resume=True,
):
    OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.to(
        device
    )

    unfreeze_all(
        model
    )

    trainable_params = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=epochs,
        )
    )

    start_epoch = 1
    best_val_loss = float("inf")
    history = []

    if (
        auto_resume
        and LAST_CKPT.exists()
    ):
        print(
            "Resume checkpoint found:",
            LAST_CKPT.resolve(),
        )

        ckpt = portable_torch_load(
            LAST_CKPT
        )

        saved_config = ckpt.get(
            "config",
            {},
        )

        expected = checkpoint_config()

        for key, value in expected.items():
            if (
                key in saved_config
                and saved_config[key] != value
            ):
                raise ValueError(
                    f"Resume checkpoint mismatch for '{key}': "
                    f"saved={saved_config[key]!r}, current={value!r}"
                )

        model.load_state_dict(
            ckpt["model"],
            strict=True,
        )

        optimizer.load_state_dict(
            ckpt["optimizer"]
        )

        scheduler.load_state_dict(
            ckpt["scheduler"]
        )

        start_epoch = (
            int(
                ckpt["epoch"]
            )
            + 1
        )

        best_val_loss = float(
            ckpt.get(
                "best_val_loss",
                float("inf"),
            )
        )

        history = ckpt.get(
            "history",
            [],
        )

        print(
            f"Resuming from epoch {start_epoch} | "
            f"best val loss={best_val_loss:.4f}"
        )

    trainable_count, total_count = (
        parameter_counts(
            model
        )
    )

    print(
        f"\nTraining: trainable parameters "
        f"{trainable_count:,} / {total_count:,} "
        f"({100.0 * trainable_count / total_count:.4f}%)"
    )

    print(
        f"Training: lr={lr:.3e}, "
        f"target epochs={epochs}, "
        f"start epoch={start_epoch}"
    )

    print(
        "Text log:",
        LOG_FILE.resolve(),
    )

    append_log(
        LOG_FILE,
        (
            f"\n=== run start / resume ===\n"
            f"variant={RUN_VARIANT}\n"
            f"target_epochs={epochs}\n"
            f"start_epoch={start_epoch}\n"
            f"lr={lr:.8e}\n"
            f"trainable={trainable_count:,}/{total_count:,}"
        ),
    )

    if start_epoch > epochs:
        print(
            f"Training already completed through epoch "
            f"{start_epoch - 1}; target is {epochs}."
        )

        return {
            "best_val_loss": best_val_loss,
            "history": history,
        }

    for epoch in range(
        start_epoch,
        epochs + 1,
    ):
        model.train()

        total_target_tokens = 0
        total_correct = 0
        loss_sum = 0.0
        max_grad_norm = 0.0

        sync_device(
            device
        )

        train_start = (
            time.perf_counter()
        )

        for batch in train_loader:
            batch = move_batch_to_device(
                batch,
                device,
            )

            optimizer.zero_grad(
                set_to_none=True,
            )

            with autocast_context(
                device
            ):
                outputs = model(
                    **batch,
                    use_cache=False,
                )

                loss = outputs.loss
                logits = outputs.logits

            loss.backward()

            if grad_clip_norm is not None:
                grad_norm = (
                    torch.nn.utils
                    .clip_grad_norm_(
                        trainable_params,
                        max_norm=grad_clip_norm,
                    )
                )

                try:
                    grad_norm_value = float(
                        grad_norm
                        .detach()
                        .cpu()
                    )

                    if math.isfinite(
                        grad_norm_value
                    ):
                        max_grad_norm = max(
                            max_grad_norm,
                            grad_norm_value,
                        )
                except Exception:
                    pass

            optimizer.step()

            shifted_logits = logits[
                :,
                :-1,
                :,
            ]

            shifted_labels = batch[
                "labels"
            ][
                :,
                1:,
            ]

            target_tokens = (
                shifted_labels.numel()
            )

            total_target_tokens += (
                target_tokens
            )

            loss_sum += (
                float(loss.item())
                * target_tokens
            )

            with torch.no_grad():
                total_correct += (
                    shifted_logits.argmax(
                        dim=-1
                    )
                    == shifted_labels
                ).sum().item()

        sync_device(
            device
        )

        train_time = (
            time.perf_counter()
            - train_start
        )

        scheduler.step()

        train_loss = (
            loss_sum
            / total_target_tokens
        )

        train_accuracy = (
            total_correct
            / total_target_tokens
        )

        train_ppl = safe_perplexity(
            train_loss
        )

        val_metrics = evaluate_lm(
            model,
            val_loader,
            device,
        )

        current_lr = (
            scheduler
            .get_last_lr()[0]
        )

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_perplexity": train_ppl,
            "train_token_accuracy": train_accuracy,
            "val_loss": val_metrics["loss"],
            "val_perplexity": val_metrics["perplexity"],
            "val_token_accuracy": val_metrics["token_accuracy"],
            "train_time_sec": train_time,
            "val_time_sec": val_metrics["time_sec"],
            "max_finite_grad_norm_before_clipping": max_grad_norm,
        }

        history.append(
            record
        )

        line = (
            f"Epoch {epoch:03d}/{epochs} | "
            f"lr={current_lr:.3e} | "
            f"train loss={train_loss:.4f} "
            f"ppl={train_ppl:.3f} "
            f"acc={train_accuracy:.4f} | "
            f"val loss={val_metrics['loss']:.4f} "
            f"ppl={val_metrics['perplexity']:.3f} "
            f"acc={val_metrics['token_accuracy']:.4f} | "
            f"train time={format_seconds(train_time)} | "
            f"val time={format_seconds(val_metrics['time_sec'])} | "
            f"max grad norm={max_grad_norm:.3e}"
        )

        print(
            line
        )

        append_log(
            LOG_FILE,
            line,
        )

        is_new_best = (
            val_metrics["loss"]
            < best_val_loss
        )

        if is_new_best:
            best_val_loss = (
                val_metrics["loss"]
            )

        checkpoint_common = {
            "epoch": epoch,
            "model": model.state_dict(),
            "val": val_metrics,
            "best_val_loss": best_val_loss,
            "config": checkpoint_config(),
            "history": history,
            "saved_from_device": str(device),
        }

        if is_new_best:
            atomic_torch_save(
                checkpoint_common,
                BEST_CKPT,
            )

        atomic_torch_save(
            {
                **checkpoint_common,
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
            },
            LAST_CKPT,
        )

        with HISTORY_FILE.open(
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(
                history,
                f,
                indent=2,
            )

    done_message = (
        f"DONE | "
        f"best validation loss={best_val_loss:.4f} | "
        f"last completed epoch={epochs}"
    )

    print(
        done_message
    )

    append_log(
        LOG_FILE,
        done_message,
    )

    return {
        "best_val_loss": best_val_loss,
        "history": history,
    }


## 7. Untouched pretrained Pythia-31M baseline

No baseline retraining is performed.

The final public pretrained checkpoint is evaluated directly on the same local held-out Pile validation subset that will be used throughout recovery.

This provides the reference:

\[
L_{\text{pretrained}},
\qquad
\mathrm{PPL}_{\text{pretrained}},
\qquad
Acc_{\text{pretrained}}.
\]


In [10]:
# ============================================================
# Untouched pretrained Pythia-31M baseline
# ============================================================

standard_model = model.to(
    device
)

standard_model.config.use_cache = False

print(
    "\n=== Untouched pretrained Pythia-31M "
    "/ normal LayerNorm + Softmax ==="
)

print_trainable_parameters(
    standard_model
)

standard_metrics = print_eval(
    standard_model,
    val_loader,
    device,
    title="Pretrained Pythia-31M",
    split_name="Validation",
)



=== Untouched pretrained Pythia-31M / normal LayerNorm + Softmax ===
Trainable: 30,494,720 / 30,494,720 (100.0000%)

[Pretrained Pythia-31M]
Validation: loss=4.4852, ppl=88.693, next-token acc=0.2820, time=12.2s


## 8. One-shot Full LENS recovery

A fresh copy of the same final pretrained Pythia-31M checkpoint is loaded.

Then, in one shot:

1. every `LayerNorm` is replaced by LearnedNorm;
2. every GPT-NeoX causal attention Softmax is replaced by Direct PureMLP attention;
3. the whole model is unfrozen immediately;
4. the model is retrained for 50 epochs on the Pile causal-LM objective.

The immediate pre-recovery metrics are measured before any optimizer step, so the replacement shock is visible directly.


In [ ]:
# ============================================================
# Install Full LENS in one shot and retrain on The Pile
# ============================================================

# Free the untouched baseline model before constructing the recovery model.
standard_model = standard_model.to(
    "cpu"
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Fresh copy of the exact same pretrained checkpoint.
model = load_local_pythia31m()


# ------------------------------------------------------------
# Replace BOTH operator families together.
# ------------------------------------------------------------
replaced_norms = (
    replace_all_layernorms(
        model,
        hidden_dim=LEARNED_NORM_HIDDEN,
    )
)

replaced_attn = (
    replace_all_attention_mapping(
        model,
        row_hidden_dim=ATTENTION_ROW_HIDDEN,
        element_hidden_dim=ATTENTION_ELEMENT_HIDDEN,
    )
)


model.config.use_cache = False

model = model.to(
    device
)

unfreeze_all(
    model
)


print(
    f"Replaced {len(replaced_norms)} LayerNorm modules."
)

for name in replaced_norms:
    print(
        "  LN:",
        name,
    )


print(
    f"Replaced Softmax attention mapping in "
    f"{len(replaced_attn)} GPT-NeoX attention modules."
)

for name in replaced_attn:
    print(
        "  Attn:",
        name,
    )


print(
    "\n=== Full LENS installed: before recovery ==="
)

print_trainable_parameters(
    model
)


pre_recovery_metrics = print_eval(
    model,
    val_loader,
    device,
    title=(
        "Full LENS Pythia-31M "
        "immediately after replacement"
    ),
    split_name="Validation",
)


print(
    "\n=== Joint Full-LENS Pile recovery ==="
)

training_result = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=RECOVERY_EPOCHS,
    lr=RECOVERY_LR,
    weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM,
    auto_resume=AUTO_RESUME,
)


if not BEST_CKPT.exists():
    raise FileNotFoundError(
        f"Best checkpoint was not created: "
        f"{BEST_CKPT}"
    )


load_model_checkpoint(
    model,
    BEST_CKPT,
    device,
    strict=True,
)


final_metrics = print_eval(
    model,
    val_loader,
    device,
    title="Best Full-LENS Pythia-31M / Pile",
    split_name="Validation",
)


print(
    "\n=== Summary ==="
)

print(
    f"Pretrained standard | "
    f"loss={standard_metrics['loss']:.4f} | "
    f"ppl={standard_metrics['perplexity']:.3f} | "
    f"acc={standard_metrics['token_accuracy']:.4f}"
)

print(
    f"Immediate Full LENS | "
    f"loss={pre_recovery_metrics['loss']:.4f} | "
    f"ppl={pre_recovery_metrics['perplexity']:.3f} | "
    f"acc={pre_recovery_metrics['token_accuracy']:.4f}"
)

print(
    f"Recovered Full LENS | "
    f"loss={final_metrics['loss']:.4f} | "
    f"ppl={final_metrics['perplexity']:.3f} | "
    f"acc={final_metrics['token_accuracy']:.4f}"
)

print(
    f"Recovered loss delta vs pretrained: "
    f"{final_metrics['loss'] - standard_metrics['loss']:+.4f}"
)

print(
    f"Recovered next-token accuracy delta vs pretrained: "
    f"{100.0 * (final_metrics['token_accuracy'] - standard_metrics['token_accuracy']):+.3f} pp"
)

print(
    "Best checkpoint:",
    BEST_CKPT.resolve(),
)

print(
    "Resume checkpoint:",
    LAST_CKPT.resolve(),
)

print(
    "Epoch text log:",
    LOG_FILE.resolve(),
)


Replaced 13 LayerNorm modules.
  LN: gpt_neox.layers.0.input_layernorm
  LN: gpt_neox.layers.0.post_attention_layernorm
  LN: gpt_neox.layers.1.input_layernorm
  LN: gpt_neox.layers.1.post_attention_layernorm
  LN: gpt_neox.layers.2.input_layernorm
  LN: gpt_neox.layers.2.post_attention_layernorm
  LN: gpt_neox.layers.3.input_layernorm
  LN: gpt_neox.layers.3.post_attention_layernorm
  LN: gpt_neox.layers.4.input_layernorm
  LN: gpt_neox.layers.4.post_attention_layernorm
  LN: gpt_neox.layers.5.input_layernorm
  LN: gpt_neox.layers.5.post_attention_layernorm
  LN: gpt_neox.final_layer_norm
Replaced Softmax attention mapping in 6 GPT-NeoX attention modules.
  Attn: gpt_neox.layers.0.attention
  Attn: gpt_neox.layers.1.attention
  Attn: gpt_neox.layers.2.attention
  Attn: gpt_neox.layers.3.attention
  Attn: gpt_neox.layers.4.attention
  Attn: gpt_neox.layers.5.attention

=== Full LENS installed: before recovery ===
Trainable: 30,694,918 / 30,694,918 (100.0000%)

[Full LENS Pythia-31M imm

## Notes

- Model: `EleutherAI/pythia-31m`, revision `step143000`.
- Local model path: `model/pythia31m`.
- Training-data source:
  - `EleutherAI/pile-duped-pythia-random-sampled`
  - HF-hosted, pre-tokenized random sample from the standard ("duped")
    Pythia training data.
- Local packed tensors: `data/Pile`.
- Sequence length: `512`.
- Default local pilot subset:
  - 64,000 packed training sequences;
  - 5,000 disjoint packed validation sequences.
- Recovery: 50 epochs.
- Full LENS is installed in one shot:
  - every LayerNorm -> LearnedNorm;
  - every causal attention Softmax -> Direct PureMLP.
- LearnedNorm:
  - `D -> 32 -> 2`;
  - `a` clamped to `[0.05, 12]`;
  - `b` unclamped.
- PureMLP:
  - row state: `512 -> 32 -> 1`;
  - element mapper: `2 -> 16 -> 1`;
  - final Sigmoid;
  - no attention Softmax;
  - no `exp` / `exp2`;
  - every causal row computes its own valid-key count;
  - divisor: `2 ** floor(log2(valid_count_row))`.
- Best recovery checkpoint is selected by lowest validation causal-LM loss.
- Validation output reports:
  - causal-LM loss;
  - perplexity;
  - next-token accuracy.
- This version never accesses `the-eye.eu`, so the expired certificate in the
  old raw-Pile loader is no longer involved.
